# InsightFace サンプル

## InsightFace — 顔検出と顔分析

このノートブックは **InsightFace** を使った顔検出と顔分析のサンプルです。

```
画像（URL またはファイル）→ InsightFace → アノテーション済み画像 + バウンディングボックス + ランドマーク + 年齢/性別
```

InsightFace は画像内のすべての顔を検出し、各顔について以下の情報を提供します：
- **バウンディングボックス** — 顔の位置
- **ランドマーク** — 5 点（両目・鼻・口角）
- **年齢・性別** — 推定年齢と性別

### 使用可能なモデル

| モデル | 説明 | 速度 |
|-------|------|------|
| `buffalo_l` | 最高精度（検出 + 位置合わせ + 認識 + 3D + 年齢/性別） | 最も遅い |
| `buffalo_m` | 中間精度、年齢/性別あり | — |
| `buffalo_s` | 最速、年齢/性別なし | 最も速い |
| `buffalo_sc` | 超軽量、監視カメラ向け | — |

writefile セル内の `MODEL_NAME` を変更することでモデルを切り替えられます。

📄 [InsightFace GitHub](https://github.com/deepinsight/insightface)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/CV-Samples/face"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!ls


In [ ]:
# Download sample images from GitHub
import os

BASE_URL = "https://raw.githubusercontent.com/mastnk/cv-samples/main/face"
IMAGE_FILES = [
    "sunriseforever-family-7638971_640.jpg",
]
for fname in IMAGE_FILES:
    url  = f'{BASE_URL}/{fname}'
    dest = f'{PROJECT_PATH}/{fname}'
    if not os.path.exists(dest):
        os.system(f'wget -q "{url}" -O "{dest}"')
        print(f'Downloaded: {fname}')
    else:
        print(f'Already exists: {fname}')

%cd "{PROJECT_PATH}"
!ls


In [ ]:
# Install InsightFace
!pip install insightface onnxruntime -q


## 独自画像を使うには

画像を用意する方法は 2 つあります。

**① URL を指定する**  
スクリプト実行時に `--url` フラグに直接画像 URL を渡します：
```
%run insightface.py --url https://cdn.pixabay.com/photo/2021/04/17/18/16/asian-6186466_640.jpg
```

**② 画像ファイルを `PROJECT_PATH/` に置く**  
画像ファイルを `PROJECT_PATH/` に配置してから `--file` または `--dir` で実行します。

アップロードは **Google Drive** 経由が簡単です：  
[drive.google.com](https://drive.google.com) を開き、`マイドライブ / CV-Samples / face/` に移動してファイルをドラッグ＆ドロップするだけ。  
マウント済みの Drive を通じて Colab から即座にアクセスできます。追加のアップロード手順は不要です。

```
%run insightface.py --file your_image.jpg   # ファイル 1 枚
%run insightface.py --dir  .                # フォルダ内の全画像
```

## モデルを選択するには

下の writefile セル内の `MODEL_NAME` を変更してモデルを切り替えます。  
`MODEL_NAME = ...` の行が複数ある場合、**有効になるのは最後の 1 行だけ**です。

```python
# MODEL_NAME = 'buffalo_l'   # most accurate (det + align + recog + 3D + age/gender)
# MODEL_NAME = 'buffalo_m'   # medium, age/gender included
# MODEL_NAME = 'buffalo_sc'  # tiny model, for CCTV / low-res
MODEL_NAME   = 'buffalo_s'   # fastest, no age/gender  ← active
```

モデルが大きいほど精度は高くなりますが、実行時間も長くなります。まずは `buffalo_s` で試してみましょう。  
年齢・性別推定が必要な場合は `buffalo_l` または `buffalo_m` を使ってください。

In [ ]:
# Save this cell as a Python file (Execute after editing)
%%writefile face.py
"""InsightFace Face Detection & Analysis — command-line interface.

Usage:
  %run insightface.py --url  <image_url>  [--disp]
  %run insightface.py --file <image_path> [--disp]
  %run insightface.py --dir  <image_dir>  [--disp]
"""

import argparse
import glob
import os

import cv2
import numpy as np
from PIL import Image
import requests
from io import BytesIO
from insightface.app import FaceAnalysis

# ---- IPython display (works when executed via %run in Colab) -----
try:
    from IPython.display import display as ipy_display
    _has_ipy = True
except ImportError:
    _has_ipy = False

# ---- Configuration -----------------------------------------------
MODEL_NAME   = 'buffalo_s'   # fastest, no age/gender
# MODEL_NAME = 'buffalo_m'   # medium, age/gender included
# MODEL_NAME = 'buffalo_l'   # most accurate (det + align + recog + 3D + age/gender)
# MODEL_NAME = 'buffalo_sc'  # tiny model, for CCTV / low-res

PROJECT_PATH = '/content/drive/MyDrive/CV-Samples/face'

# ---- Model loading -----------------------------------------------
app = FaceAnalysis(
    name=MODEL_NAME,
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
)
app.prepare(ctx_id=0, det_size=(640, 640))

# ---- Display helper ----------------------------------------------
def show(annotated: np.ndarray, label: str, disp: bool) -> None:
    """When --disp is active, display the annotated image then print the filename/URL."""
    if disp:
        if _has_ipy:
            ipy_display(Image.fromarray(annotated[:, :, ::-1]))  # BGR -> RGB
        print(label)

# ---- Helper: print face info ------------------------------------
def print_face_info(faces):
    """Print detection score, age, and gender for each detected face."""
    print(f'  Faces detected: {len(faces)}')
    for i, face in enumerate(faces):
        info = f'  Face {i+1}: det_score={face.det_score:.2f}'
        if hasattr(face, 'age') and face.age is not None:
            info += f'  age={int(face.age)}'
        if hasattr(face, 'gender') and face.gender is not None:
            gender_str = 'M' if face.gender == 1 else 'F'
            info += f'  gender={gender_str}'
        print(info)

# ---- Functions ---------------------------------------------------
def analyze_url(url: str, disp: bool = False):
    """Download an image from a URL and analyze faces."""
    image = np.array(Image.open(BytesIO(requests.get(url).content)).convert('RGB'))[:, :, ::-1]  # RGB -> BGR
    faces = app.get(image)
    annotated = app.draw_on(image, faces)
    show(annotated, url, disp)
    print_face_info(faces)
    return faces


def analyze_file(path: str, disp: bool = False):
    """Analyze faces in a single local image file."""
    image = cv2.imread(path)
    faces = app.get(image)
    annotated = app.draw_on(image, faces)
    show(annotated, path, disp)
    print_face_info(faces)
    return faces


def analyze_dir(directory: str, disp: bool = False):
    """Analyze faces in all images in a directory."""
    exts = ['jpg', 'jpeg', 'png', 'bmp', 'JPG', 'JPEG', 'PNG', 'BMP']
    filepaths = []
    for ext in exts:
        filepaths += sorted(glob.glob(os.path.join(directory, f'*.{ext}')))

    if not filepaths:
        print(f'No images found in: {directory}')
        return

    for path in filepaths:
        analyze_file(path, disp)
        print()


# ---- Argument parsing --------------------------------------------
parser = argparse.ArgumentParser(description='InsightFace Face Detection & Analysis')
group  = parser.add_mutually_exclusive_group(required=True)
group.add_argument('--url',  type=str, help='Image URL to analyze')
group.add_argument('--file', type=str, help='Path to a single image file')
group.add_argument('--dir',  type=str, help='Directory of images to analyze')
parser.add_argument('--disp', action='store_true',
                    help='Display annotated image and filename/URL before results')
args = parser.parse_args()

# ---- Run ---------------------------------------------------------
if args.url:
    analyze_url(args.url,   disp=args.disp)
elif args.file:
    analyze_file(args.file, disp=args.disp)
elif args.dir:
    analyze_dir(args.dir,   disp=args.disp)


## `face.py` の使い方

上の `%%writefile` セルを実行すると、`face.py` が `PROJECT_PATH` に保存されます。  
`--disp` でのインライン画像表示を有効にするため、`!python` ではなく **`%run`** で実行してください。

```
%run face.py --url  <画像URL>        # リモート画像の顔分析
%run face.py --file <画像パス>       # ローカルファイル 1 枚の顔分析
%run face.py --dir  <ディレクトリ>  # フォルダ内の全画像を顔分析
```

**オプション引数**

| フラグ | デフォルト | 説明 |
|--------|-----------|------|
| `--disp` | オフ | 結果の前にアノテーション済み画像とファイル名 / URL を表示 |

**実行例**

```bash
# リモート画像の顔分析（表示あり）
%run face.py --url https://cdn.pixabay.com/photo/2021/04/17/18/16/asian-6186466_640.jpg --disp

# ファイル 1 枚の顔分析
%run face.py --file sunriseforever-family-7638971_640.jpg --disp

# フォルダ内の全画像を顔分析
%run face.py --dir . --disp
```

**出力形式（`--disp` あり）**

```
<バウンディングボックスとランドマーク付きアノテーション済み画像をインライン表示>
sunriseforever-family-7638971_640.jpg
  Faces detected: 3
  Face 1: det_score=0.98  age=34  gender=F
  Face 2: det_score=0.95  age=8   gender=M
  Face 3: det_score=0.91  age=40  gender=M
```

> **注意:** 年齢・性別は `buffalo_m` または `buffalo_l` モデルでのみ利用できます。

## 実行方法

`face.py` は `!python` ではなく **`%run`** で実行してください。Colab カーネル内で実行されるため、`--disp` でのインライン画像表示が有効になります。

| モード | フラグ | 説明 |
|--------|--------|------|
| URL から | `--url <url>` | リモート画像を取得して顔分析 |
| ファイル 1 枚 | `--file <パス>` | ローカル画像 1 枚を顔分析 |
| ディレクトリ | `--dir <パス>` | フォルダ内の全画像を顔分析 |

`--disp` を追加すると、結果の前に各アノテーション済み画像（バウンディングボックス・ランドマーク付き）とファイル名 / URL が表示されます。

> **ヒント:** 年齢・性別推定を有効にするには、writefile セルで `buffalo_l` または `buffalo_m` に切り替えてください。

In [ ]:
# Execute: analyze faces in an image from URL (with display)
%cd "{PROJECT_PATH}"
%run face.py --disp --url https://cdn.pixabay.com/photo/2021/04/17/18/16/asian-6186466_640.jpg


In [ ]:
# Execute: analyze faces in a single local image file (with display)
%cd "{PROJECT_PATH}"
%run face.py --disp --file sunriseforever-family-7638971_640.jpg


In [ ]:
# Execute: analyze faces in all images in a directory (with display)
%cd "{PROJECT_PATH}"
%run face.py --disp --dir .


💾 **ノートブックを保存するのを忘れずに！**

作業内容を保持するため、閉じる前に **ファイル → ドライブにコピーを保存** を実行してください。